# Blockchain Case Study Advisor — NPV Portfolio Optimization

> **Goal:** Which features should we build, and do they create value?
>
> This notebook connects the investment analysis (NPV) from Notebook 03
> with the ILP optimizer — so the decision table speaks the same language
> as the financial analysis.
>
> **Primary horizon: 3 years** — matching the actual investment period (2–3 year installments).
> Year-1 NPV serves as a payback check: does the feature earn back its cost quickly?
>
> **Audience:** Product Owners, Portfolio Owners, and Agile leaders. Risk Managers will find detailed risk metrics in the evidence sections.
> **Constraint:** Upfront investment only (Year 0). Linear (ILP) optimization only.

| Who reads what | Section |
|:---------------|:--------|
| **Product Owner** | Section 1 (feature economics) → Section 2 (budget recommendation) |
| **Risk Manager** | Section 1 (PI ranking) → Section 2 (horizon gap analysis) |
| **Board** | Section 2 decision table only |

## Setup — Load the blockchain scenario

In [ ]:
from fhs.application import AdvancedPortfolioService
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import show

setup = notebook_setup("blockchain")
scenario = setup.scenario

if scenario is None:
    raise RuntimeError("Scenario setup could not be initialized")
features = scenario.features
service = AdvancedPortfolioService.from_scenario(
    scenario,
    seed=scenario.seed,
    scenarios=scenario.scenarios,
)

discount_rate = scenario.discount_rate
full_budget = float(scenario.budget)
budget_levels = {
    "25% Budget": full_budget * 0.25,
    "50% Budget": full_budget * 0.50,
    "100% Budget": full_budget,
}

show.info(
    f"Source: <code>{scenario.config_path}</code><br>"
    f"Full budget: <b>EUR {full_budget:,.0f}</b> · "
    f"Discount rate: <b>{discount_rate:.0%}</b><br>"
    "Optimization objectives: <b>npv_year1</b> and <b>npv_3year</b>."
)

---

## 1) Feature NPV Comparison — Does each feature create value?

> *NPV > 0 means the feature earns more than it costs, adjusted for the time value of money.
> The discount rate sets the minimum annual return required on the investment.*
>
> **PI (Profitability Index)** = NPV / Investment.
> PI > 0 means value-creating. The higher the PI, the more value per euro invested.
> Use PI to rank features when budget is scarce.

In [ ]:
# Compute per-feature NPV scores via the Application layer
npv_rows = service.decisions.npv_feature_comparison_rows(discount_rate=discount_rate)

show.npv_comparison(npv_rows)

show.note(
    "Green = NPV > 0 (value-creating). Red = NPV < 0 (value-destroying). "
    "PI > 0 means positive return per euro. "
    "A feature with negative Year-1 NPV but positive 3-year NPV benefits from "
    "growth that only pays off over time.",
    compact=True,
)

---

## 2) ILP Optimization — Which features should we build?

We run the **same ILP solver** with two NPV objectives across three budget levels.

| Horizon | What it answers |
|:--------|:----------------|
| **3 Years** (primary) | Does the feature create value over its planned investment period? |
| **Year 1** (payback check) | Does the feature pay back within one year? |

When both horizons agree → high confidence. When they disagree → the feature
needs growth to pay off — acceptable if the business can wait.

In [ ]:
# Year-1 focus (conservative, fast payback)
ilp_year1 = service.budget_sensitivity(
    budget_levels, solver="ilp", strategy="npv_year1"
)

# 3-year focus (growth-oriented, full investment horizon)
ilp_3year = service.budget_sensitivity(
    budget_levels, solver="ilp", strategy="npv_3year"
)

# Build the combined decision table via the Application layer
decision_rows = service.decisions.npv_decision_table_rows(
    budget_levels, ilp_year1, ilp_3year, discount_rate=discount_rate
)

show.npv_decision_table(decision_rows)

---

## 3) Key Insights

### What the table reveals

- **Budget drives the decision.** The same optimizer picks different features depending on budget.
  At 25 % budget, only one feature fits — the optimizer picks the one with the best NPV per euro.
- **Time horizon matters.** A feature with negative Year-1 NPV can have positive 3-year NPV
  if growth kicks in over time. The 3-year view may include features the Year-1 view skips.
- **One table, two perspectives.** The Product Owner picks the row that matches their
  planning horizon. No separate ROI, IRR, or multi-year tables needed.
- **Upfront investment model.** All development cost is paid in Year 0. Operating costs
  are deducted from business value each year. This matches NB 03 Option A.

---

### Decision Governance — Who Decides?

The optimizer recommends which features to build. But **who approves the investment?** The answer depends on the risk profile:

| Condition | Decision Level | Rationale |
|---|---|---|
| Business Value Floor > investment cost | **Product team** decides autonomously | Conservative floor exceeds cost — low risk |
| Business Value Floor < 0, but Expected Value > cost | **Department head** approval required | Positive average, but real downside — needs oversight |
| CVaR 95% < −1× investment | **CFO / executive board** review | Worst-case losses exceed the entire investment — strategic decision |

> **These thresholds are examples.** Adapt them to your organization's risk appetite. A startup may accept higher risk autonomously; a regulated company may require board approval earlier.

**Why this matters for Product Owners:** Without clear escalation rules, every feature decision becomes a political discussion. With these thresholds, you can say: *"Our business value floor is above cost — the data supports autonomous approval."*

### How NPV objectives compare to var_floor

| Objective | Score formula | Best for |
|-----------|--------------|----------|
| `var_floor` | $\mathrm{VaR}_{95\%} - C$ | Risk-first: protect against worst-case losses |
| `npv_year1` | $\frac{BV_1 - OpEx_1}{1+r} - C$ | Fast payback: does it pay back within one year? |
| `npv_3year` | $\sum_{t=1}^{3} \frac{BV_t(1+g)^{t-1} - OpEx_t}{(1+r)^t} - C$ | Growth: does it create value over the full horizon? |

All three are **linear and additive** per feature — the ILP solver works without changes.

| Term | Definition |
|------|------------|
| **NPV** | Net Present Value — sum of discounted future cash flows minus the upfront investment. NPV > 0 = value-creating. |
| **PI** | Profitability Index — NPV per euro of invested capital. Ranks features by capital efficiency. |
| **Discount rate** | Minimum annual return required on an investment. Higher rate = stricter threshold. |
| **ILP** | Integer Linear Programming — exact solver that guarantees the optimal feature combination within a budget. |
| **Upfront investment** | All development cost paid in Year 0 (Option A from NB 03). No installment spreading. |
| **Business Value Floor 95** | The 5th percentile of the simulation. 95 % of scenarios produce a result above this floor. |

---

## Notebook Navigation

| # | Notebook | What you learn |
|:-:|----------|----------------|
| 01 | [Getting Started](01-getting-started.ipynb) | One feature, simulation basics, business value floor dashboard |
| 02 | [Blockchain Case Study](02-blockchain-case-study.ipynb) | Feature risks, portfolio baseline, board recommendation |
| 03 | [Capital Budgeting (ROI, NPV, IRR)](03-blockchain-case-study-capital-budgeting.ipynb) | Investment metrics for funding decisions |
| 04 | **Blockchain Advisor — NPV-Based Portfolio Optimization** | ← You are here |
| 05 | [Blockchain Case Study Risk](05-blockchain-case-study-risk.ipynb) | Risk on ILP-selected portfolios under fixed budgets |
| 06 | [Delivery Risk](06-blockchain-case-study-delivery-risk.ipynb) | Sprint overruns, cost simulation, budget fit |
| 07 | [Executive Decision](07-blockchain-case-study-decision.ipynb) | All dimensions in one view, combined recommendation |
| A01 | [Portfolio Optimiser (10 features)](advanced/01-portfolio-advisor.ipynb) | 10-feature portfolio, three solvers (Exact/ILP/Greedy), concentration risk |
| A02 | [Capital Budgeting at Scale](advanced/02-portfolio-risk-dashboard.ipynb) | 10-feature capital budgeting, upfront vs. installment financing |

---

*Previous: [03-blockchain-case-study-capital-budgeting.ipynb](03-blockchain-case-study-capital-budgeting.ipynb) · Next: [05-blockchain-case-study-risk.ipynb](05-blockchain-case-study-risk.ipynb)*

---
**Previous:** [NB 03: Capital Budgeting](03-blockchain-case-study-capital-budgeting.ipynb) | **Next:** [NB 05: Risk Analysis](05-blockchain-case-study-risk.ipynb)
